# 2.2 Data Preprocessing

So far we have worked with synthetic data that arrived in ready-made tensors. To apply deep learning in the real world, though, we must first extract messy data stored in arbitrary formats and preprocess it to suit our needs. The **`pandas`** library does much of this heavy lifting — this section is a crash course on its most common routines: reading a CSV file, handling missing values, and converting the result into tensors.

## 2.2.1 Reading the Dataset

**Comma-separated values (CSV)** files are ubiquitous for storing tabular (spreadsheet-like) data: each line is one record, and its fields are separated by commas. To demonstrate `pandas`, we first write a tiny CSV file by hand, `../data/house_tiny.csv`. Each row is a home; the columns are the number of rooms (`NumRooms`), the roof type (`RoofType`), and the price (`Price`). An entry of `NA` marks a value that is missing.

In [1]:
import os  # standard library: build the data directory path

In [2]:
# Create data directory and write sample CSV file
os.makedirs(os.path.join('..', 'data'), exist_ok=True)     # ../data, relative to this notebook
data_file = os.path.join('..', 'data', 'house_tiny.csv')   # this creates a file path
with open(data_file, 'w') as f:                             # open the file for writing
    # ''' or """ is used for multi-line strings
    f.write('''NumRooms,RoofType,Price
NA,NA,127500
2,NA,106000
4,Slate,178100
NA,NA,140000''')

In [3]:
import pandas as pd
data = pd.read_csv(data_file)   # 'NA' entries are parsed as NaN (missing) automatically
print(data)

   NumRooms RoofType   Price
0       NaN      NaN  127500
1       2.0      NaN  106000
2       4.0    Slate  178100
3       NaN      NaN  140000


## 2.2.2 Data Preparation

In supervised learning we train models to predict a designated **target** value given some set of **input** values, so our first step is to separate the target column from the input columns — here via integer-location indexing (`iloc`), though `pandas` also supports selecting by name.

Notice above that `pandas` replaced every `NA` entry with a special **`NaN`** (not a number) marker; the same happens for any empty field (e.g. `3,,270000`). These **missing values** are, in the book's words, "the bed bugs of data science" — a persistent nuisance you'll meet again and again. Depending on context they're handled by **imputation** (estimating a replacement value) or **deletion** (dropping the offending rows or columns).

For a categorical field like `RoofType`, one common heuristic treats `NaN` itself as a category: `get_dummies` turns the column into indicator columns `RoofType_Slate` and `RoofType_nan`, one per observed value (`dummy_na=True` includes the missing category).

In [4]:
inputs, targets = data.iloc[:, 0:2], data.iloc[:, 2]  # columns 0:2 are features, column 2 (Price) is the target
inputs = pd.get_dummies(inputs, dummy_na=True)         # one-hot encode RoofType; NaN becomes its own indicator column
print(inputs)

   NumRooms  RoofType_Slate  RoofType_nan
0       NaN           False          True
1       2.0           False          True
2       4.0            True         False
3       NaN           False          True


For the remaining missing **numerical** values in `NumRooms`, a common heuristic is to fill each one with the mean of its column.

In [5]:
inputs = inputs.fillna(inputs.mean())  # mean imputation: replace each NaN with its column's average
print(inputs.mean())                   # the fill values actually used, for reference
print(inputs)

NumRooms          3.00
RoofType_Slate    0.25
RoofType_nan      0.75
dtype: float64
   NumRooms  RoofType_Slate  RoofType_nan
0       3.0           False          True
1       2.0           False          True
2       4.0            True         False
3       3.0           False          True


## 2.2.3 Conversion to the Tensor Format

Every entry in `inputs` and `targets` is now numerical (including the booleans produced by `get_dummies`), so both can be loaded straight into tensors, ready for the tensor operations from §2.1.

In [6]:
import torch

X = torch.tensor(inputs.to_numpy(dtype=float))  # feature matrix
y = torch.tensor(targets.to_numpy(dtype=float))  # target vector
X, y

(tensor([[3., 0., 1.],
         [2., 0., 1.],
         [4., 1., 0.],
         [3., 0., 1.]], dtype=torch.float64),
 tensor([127500., 106000., 178100., 140000.], dtype=torch.float64))

## 2.2.4 Summary

- `pandas.read_csv` loads tabular data into a `DataFrame`; any `NA` (or empty) field becomes `NaN`.
- Split **inputs** from **targets** first (e.g. with `iloc`) so imputation and encoding never touch the column you're trying to predict.
- Missing values are handled by **imputation** (estimate a replacement — the column mean for numeric fields, or treat `NaN` as its own category for categorical ones) or by **deletion** of the offending rows/columns.
- `pd.get_dummies(..., dummy_na=True)` turns a categorical column into one indicator column per observed value, including missing.
- Once every column is numerical, `torch.tensor(df.to_numpy(dtype=float))` hands the data off to PyTorch.
- Real datasets are messier than this toy example — data spread across multiple files, non-tabular types (text, images, audio), outliers and sensor errors — and are worth inspecting visually (e.g. with `seaborn` or `matplotlib`) before they ever reach a model.